In [1]:
import chaospy
def generate_draws(n=3, d=2, rule="random"):
    '''Generate draws for income and asset returns
    Parameters:
        n: number of draws (scalar)
        d: number of dimensions (scalar)
        rule: quadrature rule for income and asset returns
            (str) "legendre", "random", "halton", "sobol"
    Returns:
        w: weights for quadrature (1 x N) or weight w=1/n for random draws (scalar)
        x: draws for income and asset returns (d x n)
        for muldimensional case, quadrature nodes are tensor product of 1D nodes
        with int(n**(1/d)) nodes in each dimension
    '''
    distribution = chaospy.Iid(chaospy.Uniform(0, 1), d)
    if rule == 'legendre':
        order = int(n**(1/d)) - 1
        x, w = chaospy.generate_quadrature(order, distribution, rule=rule, sparse=False)
    else:
        x = chaospy.generate_samples(n, domain=d, rule=rule)  
        w = np.full((1, x.shape[1]), 1/x.shape[1])  # Equal weight for Monte Carlo
    return w, x

In [2]:
import numpy as np
from scipy.stats import norm  
def inverse_multinormal(u, mu=None, Sigma=None):
    '''Inverse transform sampling for multivariate normal
    Parameters:
        u: uniform random numbers (d x n)
        mu: mean vector (d x 1)
        Sigma: covariance matrix (d x d)
    Returns:
        x: draws from multivariate normal (d x n) with x~N(mu, Sigma)
    '''
    x = norm.ppf(u)  # inverse cdf of standard normal
    if Sigma is not None:
        L = np.linalg.cholesky(Sigma)  
        x = L @ x # 
    if mu is not None:
        mu = np.array(mu).reshape(-1,1)
        x += mu
    return x

In [ ]:
def build_sigma(sigma = [1,.1], c=0): 
    '''build d-dimensional (dxd) covariance matrix Sigma with correlation c and std deviations sigma
    Parameters:
        sigma: list of std deviations for d random variables
        c: correlation coefficient between random variables (default=0) can only create negaive correlation for d=2'''
    sigma=np.array(sigma).reshape(-1,1); 
    d=len(sigma);
    if (c<0.0) and not (d==2):
        raise RuntimeError('build_sigma can only create negaive correlation for d=2')
    corr=(1-c)*np.identity(d) + c*np.ones((d,d)); # correlation martrix 
    Sigma=sigma*corr*sigma.T # Covariance matrix  
    return Sigma


In [24]:
beta = 0.96
gamma = 2
mu_y = 0
sigma_y = 0.1

mu_R= 1+ np.linspace(0.02,0.04, 2)
sigma = np.linspace(0.05, 0.15, 2)
Sigma_R = build_sigma(sigma, c=0)
rho_y = np.zeros((len(mu_R),1))
bnd_x = [0.1, 10]
n_x = 50
n_rand = 100
N = len(mu_R)
n_choices = N + 1
x0 = np.linspace(bnd_x[0], bnd_x[1], n_x).reshape(n_x, 1)
mu_R = np.array(mu_R).reshape(N, 1)
mu = np.vstack([[mu_y], mu_R])
sigma = np.block([
            [sigma_y**2, rho_y.T @ Sigma_R],
            [Sigma_R @ rho_y, Sigma_R]
        ])  

w, u = generate_draws(n_rand, N+1, rule="random")
x = inverse_multinormal(u, mu, sigma)
y = np.exp(x[0,:]).reshape(1,-1)
R = x[1:,:]

In [40]:
np.shape(w), np.shape(y), np.shape(R)

((1, 100), (1, 100), (2, 100))

In [41]:
print(w)

[[0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01
  0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01
  0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01
  0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01
  0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01
  0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01
  0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01
  0.01 0.01]]


In [17]:
def utility(c): 
    '''CRRA utility function'''
    if gamma!=1:
        return (c**(1-gamma)-1)/(1-gamma)+1
    else:
        return np.log(c)+1    

In [18]:
# %% Returns
def returns(w):
    '''Compute portfolio returns for a given choice of weights w'''
    
    # Ensure w is a column vector (N x 1)
    w = np.array(w).reshape(N, 1)  # Portfolio weights (N x 1)

    # Compute portfolio return: R_t = w' * R_t+1
    R_portfolio = w.T @ R  # Shape: (1 x n_rand)

    return R_portfolio

In [42]:
returns(w)

ValueError: cannot reshape array of size 100 into shape (2,1)

In [20]:
def transition(x, c, w):
    '''Compute next-period cash-on-hand x_{t+1} given x_t, c_t, and portfolio choice w_t'''
    
    # Ensure inputs are in correct shape
    x = np.array(x).reshape(-1, 1)  # Reshape x to (n_x,1) for vectorized computation
    c = np.array(c).reshape(-1, 1)  # Reshape c to (n_x,1)
    
    # Compute portfolio returns for given weights w (1 x n_rand)
    R_next = returns(w)  
    
    # Compute next-period cash-on-hand for all Monte Carlo draws (n_x x n_rand)
    x_next = (x - c) @ R_next + y  # Broadcasting ensures correct shapes
    return x_next  # Shape: (n_x, n_rand)

In [21]:
def vf(choices, interpV):
    '''Compute the value function for given choices (c, w) and interpolated value function'''
    
    # Parse choices
    c = choices[0]  # Consumption choice (scalar)
    w = choices[1:]  # Portfolio allocation (vector)

    # Compute next-period cash-on-hand
    x_next = transition(x0, c, w)  # Shape: (n_x, n_rand)

    # Compute expected future value by taking expectation over next-period states
    EV = np.sum(w * interpV(x_next), axis=1, keepdims=True)  # Shape: (n_x, 1)

    # Compute total value function
    return utility(c) + beta * EV  # Shape: (n_x, 1)

In [39]:
c = np.array(c).reshape(-1, 1)

NameError: name 'c' is not defined

In [23]:
from scipy.optimize import minimize
from scipy import interpolate # Interpolation routines
def bellman(V0):
    '''Bellman operator: Computes updated value function and optimal policy choices'''    
    # Create interpolated value function from V0
    # interpV = interpolate.interp1d(self.x0[:, 0], V0, bounds_error=False, kind='linear', fill_value='extrapolate')
    interpV = interpolate.interp1d(x0[:, 0], V0, kind='cubic', bounds_error=False, fill_value='extrapolate')

    # Initialize storage for new value function and policy choices
    V1 = np.zeros_like(V0)  # New value function
    policy = np.zeros((n_x, 1+ N))  # Optimal policy function (0: consumption, 1:N portfolio weights)

    # Loop over each grid point in the state space
    for i, x in enumerate(x0.flatten()):  # Ensure x is a scalar       
        # Define objective function: NEGATIVE because we maximize
        obj = lambda choices: -vf(choices, interpV)[i]
        
        # Initial guess: Consume half of cash and equal portfolio weights
        if i==0:  # At the first grid point
            c_init = float(x)  # Assume consuming all cash (credit limit)
            w_init = np.ones(N) / N  # Equal allocation across assets
            choices_init = np.concatenate(([c_init], w_init))  # Correct concatenation  
        else:
            choices_init=policy[i-1, : ]*0.95 # Use previous period policy as initial guess
        
        # Constraints: c in [0,x] and portfolio weights sum to 1, w_i >= 0
        cons = [
            {'type': 'ineq', 'fun': lambda choices: x - choices[0]},           # c <= x
            {'type': 'ineq', 'fun': lambda choices: choices[0]-0.001},         # c >= 0.001
            {'type': 'eq', 'fun':   lambda choices: np.sum(choices[1:]) - 1},  # sum(w) = 1
            {'type': 'ineq', 'fun': lambda choices: choices[1:]}               # w_i >= 0
        ]

        # Solve the optimization problem
        result = minimize(obj, choices_init, method='SLSQP', constraints=cons, options={'ftol': 1e-10, 'maxiter': 200})

        # Store the results
        V1[i] = -result.fun         # Maximum value
        policy[i, : ] = result.x    # Optimal decisions

    return V1, policy # Value function and policy functions

In [36]:
import numpy as np
from scipy.stats import lognorm
from time import process_time
def vfi_T(n_x, n_choices, maxiter = 100, callback=None):
    tic = process_time() # Start the stopwatch / counter
    V=np.zeros((n_x, maxiter+1)) # on first iteration assume consuming everything
    policy = np.zeros((n_x, n_choices, maxiter+1))  # Stores policy functions (p1, p2, ..., pn_choices)'''
    
    for t in range(maxiter-1, 0, -1):
        V[:,t-1],policy[:,:,t-1]=bellman(V[:,t])
        if callback: callback(t,x,V, policy) # callback for making plots and plotting iterations
    else:  # when i went up to maxiter
        toc = process_time()
        print('Solved by backward induction using',round(toc-tic, 5), 'seconds')
    return V,policy

def vfi(n_x, n_choices, maxiter=100, tol=1e-6,callback=None):
    '''Solves the model using VFI (successive approximations)'''
    tic = process_time() # Start the stopwatch / counter
    V0=np.zeros(n_x) # on first iteration assume consuming everything
    for iter in range(maxiter):
        V1,policy=bellman(V0)
        if callback: callback(iter,x,V1, policy, V0) # callback for making plots
        if np.max(abs(V1-V0)) < tol:
            toc = process_time() # Stop the stopwatch / counter
            print('Solved by VFI in', iter, 'iterations using',round(toc-tic, 5), 'seconds')
            break
        V0=V1
    else:  # when i went up to maxiter
        print('No convergence: maximum number of iterations achieved!')
    return V1,policy

def iterinfo(iter,V1,c=None, V0=0):
    print('iter=', iter, '||V1-V0||', np.max(abs(V1-V0)))



In [27]:
# function for plotting
import matplotlib.pyplot as plt
def v_c_plot(x, V, policy):
    '''Illustrate solution'''
    c=policy[:,0,:] # Consumption policy
    fig1, (ax1,ax2) = plt.subplots(1,2,figsize=(8,4))
    ax1.grid(which='both', color='0.65', linestyle='-')
    ax2.grid(which='both', color='0.65', linestyle='-')
    ax1.set_title('Value function')
    ax2.set_title('Consumption policy function')
    ax1.set_xlabel('Cash on hand, x')
    ax2.set_xlabel('Cash on hand, x')
    ax1.set_ylabel('Value function')
    ax2.set_ylabel('Consumption function')
    if len(V.shape)==1: 
        V=V[:,np.newaxis]
        c=c[:,np.newaxis]
    for i in range(V.shape[1]):
        ax1.plot(x[1:],V[1:,i],color='k',alpha=0.25)
        ax2.plot(x[1:],c[1:,i],color='k',alpha=0.25)
    # add solutions
    ax1.plot(x[1:],V[1:,0],color='r',linewidth=2.5)
    ax2.plot(x[1:],c[1:,0],color='r',linewidth=2.5)
    plt.show()

In [28]:
import matplotlib.pyplot as plt
def w_plot(x, V, policy):
    '''Illustrate solution for portfolio shares'''
    w = policy[:, 1:, :]  # Portfolio shares (excluding consumption)
    
    num_assets = w.shape[1]  # Number of assets in portfolio
    fig, ax = plt.subplots(1, num_assets, figsize=(4 * num_assets, 4))  # One subplot per asset
    
    if num_assets == 1:  # Ensure ax is iterable even for a single asset
        ax = [ax]

    for j in range(num_assets):
        ax[j].grid(which='both', color='0.65', linestyle='-')
        ax[j].set_title(f'Portfolio share $w_{j+1}(x)$')
        ax[j].set_xlabel('Cash on hand, x')
        ax[j].set_ylabel(f'Portfolio share $w_{j+1}$')

        if len(V.shape) == 1:  # Ensure proper shape handling
            V = V[:, np.newaxis]
            w = w[:, :, np.newaxis]

        for i in range(V.shape[1]):  # Iterate over solution iterations
            ax[j].plot(x[1:], w[1:, j, i], color='k', alpha=0.25)  # Plot historical solutions
        
        # Highlight the first iteration in red
        ax[j].plot(x[1:], w[1:, j, 0], color='r', linewidth=2.5)

    plt.show()

In [38]:
V, policy = vfi_T(n_x, n_choices, maxiter=100); 
v_c_plot(x0, V, policy);
w_plot(x0, V, policy);

ValueError: operands could not be broadcast together with shapes (2,) (50,100) 